# Summary of EDA

- it is a telecommunication company dataset containing customer features (e.g. services used, loyalty time, etc..) and the goal is to predict customer churning,
- in total, the table contains 7043 rows and 21 columns, out of which 1 is the target column and 1 is a customer ID column (not useful for prediction),
- 16 + 1 categorical columns, and 3 numerical columns can be identified (not counting the ID column),
- out of the 3 numerical columns, 1 is of type integer (tenure), while the other 2 are floats (monthly and total charges),
- in the total charges column, there are 11 only-space values, which represent NaN values,
- a lot of categorical features, such as the contract type feature, are already very good indicators for churning,
- the tenure column has an approximately uniform distribution (with elevated densities at the domain ends), the monthly charges column is multimodal, while the total charges follows either an exponential or a power-law distribution,
- the tenure value creates linear boundaries for the total charges column,
- customers with a high tenure value are unlikely to churn (ROC-AUC = 0.74),
- customers with low monthly charges are unlikely to churn (ROC-AUC = 0.62),
- only the total charges feature contains outlier values (z-score > 2.7), but these should be kept, because they provide useful information and are not erroneous datapoints,
- due to the large number of categorical features, Multiple Correspondence Analysis (MCA) is best suited for dimensionality reduction,
- the information of whether a customer has internet survice or not is contained redundantly in multiple columns,
- other categorical columns add equal total variances to the MCA, meaning that all categorical features should be used for training, due to their approximate orthogonality,
- churn separation along the first two MCA components works fairly well (ROC-AUC of 0.74 and 0.74, respectively).

# Data Processing Tasks Before ML Model Training

- load the dataset, remove the customer ID column,
- in the total charges column, replace empty (single space) cells with NaN values and then fill those in with the median,
- use one-hot encoding on the categorical columns,
- maybe transform the monthly and total charges columns using logarithm,
- standardize the numerical columns.

Things that are not necessary and/or contraproductive:

- we shouldn't filter for outliers, since there aren't any erroneous rows,
- dimensionality reduction would result in a large loss of information (as seen from the explained variance plot of the multiple correspondence analysis).

In [7]:
import warnings
import pickle

import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelBinarizer, OneHotEncoder, RobustScaler
from sklearn.impute import KNNImputer

pd.set_option("display.max_columns", None)

A pipeline config dictionary should be created for the processing of the dataset. This could be done in a more advanced way with other sklearn functionalities, such as the `Pipeline`, `ColumnTransformer`, `FunctionTransformer` and `TransformerMixin` classes, but due to the presence of custom transformations (log1p transformation, " " replacement in the total charges column, dtype recasting, etc...) this approach seemed less cumbersome.

In [2]:
# Create a config dictionary used for reproducing the preprocessing
config = {
    "columns_to_remove": ["customerID", ],
    "binary_columns": [
        (["Partner", "Dependents", "PhoneService", "PaperlessBilling", "Churn"], LabelBinarizer().fit(["Yes", "No"])),
        (["gender", ], LabelBinarizer().fit(["Female", "Male"]))
    ],
    "multiclass_columns": [
        (
            ["MultipleLines", ], 
            OneHotEncoder(sparse_output=False).fit([["No", ], ["Yes", ], ["No phone service"], ])
        ),
        (
            ["OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"], 
            OneHotEncoder(sparse_output=False).fit([["No", ], ["Yes", ], ["No internet service", ], ])
        ),
        (
            ["InternetService", ],
            OneHotEncoder(sparse_output=False).fit([["DSL", ], ["Fiber optic", ], ["No", ], ])            
        ),
        (
            ["Contract", ],
            OneHotEncoder(sparse_output=False).fit([["Month-to-month", ], ["One year", ], ["Two year", ], ])            
        ),
        (
            ["PaymentMethod", ],
            OneHotEncoder(sparse_output=False).fit([
                ["Bank transfer (automatic)", ],
                ["Credit card (automatic)", ],
                ["Electronic check", ], 
                ["Mailed check", ], 
            ])            
        ),
    ],
    "imputers": list(),  # will be populated later
    "numeric_scalers": list(),  # will be populated later
}

Perform transformations and update the config dictionary, if necessary, with the trained transformers (imputers and scalers).

In [3]:
# Load the dataset
dataset = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

# Remove the unnecessary columns
dataset.drop(columns=config["columns_to_remove"], inplace=True)

# Encode binary columns
for binary_column_list, column_encoder in config["binary_columns"]:
    for binary_column_name in binary_column_list:
        dataset[binary_column_name] = column_encoder.transform(dataset[binary_column_name])

# Encode multiclass columns
for multiclass_column_list, column_encoder in config["multiclass_columns"]:
    for multiclass_column_name in multiclass_column_list:

        # A warning is thrown due to the OneHotEncoder config...
        # Just ignore it.
        with warnings.catch_warnings(action="ignore"):
            encoding_matrix = column_encoder.transform(dataset[[multiclass_column_name, ]])
        
        new_column_names = [f"{multiclass_column_name}:{cat}" for cat in column_encoder.categories_[0]]
        encoding_df = pd.DataFrame(data=encoding_matrix, columns=new_column_names)

        dataset.drop(columns=[multiclass_column_name], inplace=True)
        dataset = pd.concat([dataset, encoding_df], axis=1)

# Impute missing elements
dataset["TotalCharges"] = dataset["TotalCharges"].map(lambda x: float(x) if x != " " else float("NaN"))
total_charges_imputer = KNNImputer().fit(dataset[["TotalCharges", ]])

config["imputers"].append((["TotalCharges", ], total_charges_imputer))

dataset["TotalCharges"] = total_charges_imputer.transform(dataset[["TotalCharges", ]])

# Standardize
numeric_columns = ["tenure", "MonthlyCharges", "TotalCharges"]
transforms = [None, "log1p", "log1p"]
for column_name, transform_name in zip(numeric_columns, transforms):

    transformed_column = dataset[[column_name, ]].copy()
    if transform_name == "log1p":
        transformed_column = np.log1p(transformed_column)
    elif transform_name is None:
        pass
    else:
        raise Exception("Invalid transformation name!")

    scaler = RobustScaler().fit(transformed_column)

    config["numeric_scalers"].append((column_name, transform_name, scaler))
    
    dataset[column_name] = scaler.transform(transformed_column)

Take a look at the dataset from different angles.

In [4]:
dataset

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,Churn,MultipleLines:No,MultipleLines:No phone service,MultipleLines:Yes,OnlineSecurity:No,OnlineSecurity:No internet service,OnlineSecurity:Yes,OnlineBackup:No,OnlineBackup:No internet service,OnlineBackup:Yes,DeviceProtection:No,DeviceProtection:No internet service,DeviceProtection:Yes,TechSupport:No,TechSupport:No internet service,TechSupport:Yes,StreamingTV:No,StreamingTV:No internet service,StreamingTV:Yes,StreamingMovies:No,StreamingMovies:No internet service,StreamingMovies:Yes,InternetService:DSL,InternetService:Fiber optic,InternetService:No,Contract:Month-to-month,Contract:One year,Contract:Two year,PaymentMethod:Bank transfer (automatic),PaymentMethod:Credit card (automatic),PaymentMethod:Electronic check,PaymentMethod:Mailed check
0,0,0,1,0,-0.608696,0,1,-0.919468,-1.703665,0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1,1,0,0,0,0.108696,1,0,-0.228114,0.133600,0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
2,1,0,0,0,-0.586957,1,1,-0.288404,-1.139562,1,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
3,1,0,0,0,0.347826,0,0,-0.547698,0.121937,0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
4,0,0,0,0,-0.586957,1,1,0.005366,-0.989818,1,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,1,0,1,1,-0.108696,1,1,0.202239,0.156835,0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
7039,0,0,1,1,0.934783,1,1,0.415304,0.740632,0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
7040,0,0,1,1,-0.391304,0,1,-0.928391,-0.622642,0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
7041,1,1,1,0,-0.543478,1,1,0.060544,-0.677026,1,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0


In [5]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 41 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   gender                                   7043 non-null   int64  
 1   SeniorCitizen                            7043 non-null   int64  
 2   Partner                                  7043 non-null   int64  
 3   Dependents                               7043 non-null   int64  
 4   tenure                                   7043 non-null   float64
 5   PhoneService                             7043 non-null   int64  
 6   PaperlessBilling                         7043 non-null   int64  
 7   MonthlyCharges                           7043 non-null   float64
 8   TotalCharges                             7043 non-null   float64
 9   Churn                                    7043 non-null   int64  
 10  MultipleLines:No                         7043 no

In [6]:
dataset.describe()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,Churn,MultipleLines:No,MultipleLines:No phone service,MultipleLines:Yes,OnlineSecurity:No,OnlineSecurity:No internet service,OnlineSecurity:Yes,OnlineBackup:No,OnlineBackup:No internet service,OnlineBackup:Yes,DeviceProtection:No,DeviceProtection:No internet service,DeviceProtection:Yes,TechSupport:No,TechSupport:No internet service,TechSupport:Yes,StreamingTV:No,StreamingTV:No internet service,StreamingTV:Yes,StreamingMovies:No,StreamingMovies:No internet service,StreamingMovies:Yes,InternetService:DSL,InternetService:Fiber optic,InternetService:No,Contract:Month-to-month,Contract:One year,Contract:Two year,PaymentMethod:Bank transfer (automatic),PaymentMethod:Credit card (automatic),PaymentMethod:Electronic check,PaymentMethod:Mailed check
count,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000
mean,0.504756,0.162147,0.483033,0.299588,0.073286,0.903166,0.592219,-0.245917,-0.134247,0.265370,0.481329,0.096834,0.421837,0.496663,0.216669,0.286668,0.438450,0.216669,0.344881,0.439443,0.216669,0.343888,0.493114,0.216669,0.290217,0.398978,0.216669,0.384353,0.395428,0.216669,0.387903,0.343746,0.439585,0.216669,0.550192,0.209144,0.240664,0.219225,0.216101,0.335794,0.228880
std,0.500013,0.368612,0.499748,0.458110,0.533902,0.295752,0.491457,0.636352,0.689978,0.441561,0.499687,0.295752,0.493888,0.500024,0.412004,0.452237,0.496232,0.412004,0.475363,0.496355,0.412004,0.475038,0.499988,0.412004,0.453895,0.489723,0.412004,0.486477,0.488977,0.412004,0.487307,0.474991,0.496372,0.412004,0.497510,0.406726,0.427517,0.413751,0.411613,0.472301,0.420141
min,0.000000,0.000000,0.000000,0.000000,-0.630435,0.000000,0.000000,-1.436660,-1.901636,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,-0.434783,1.000000,0.000000,-0.735044,-0.556180,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,1.000000,0.000000,1.000000,1.000000,0.565217,1.000000,1.000000,0.264956,0.443820,1.000000,1.000000,0.000000,1.000000,1.000000,0.000000,1.000000,1.000000,0.000000,1.000000,1.000000,0.000000,1.000000,1.000000,0.000000,1.000000,1.000000,0.000000,1.000000,1.000000,0.000000,1.000000,1.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000
max,1.000000,1.000000,1.000000,1.000000,0.934783,1.000000,1.000000,0.567837,0.814338,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


The preprocessing seems OK, so we can save the config file and the processed dataset.

In [9]:
with open("preprocessing_config.pkl", "wb") as f:
    pickle.dump(config, f)

with open("preprocessed_dataset.pkl", "wb") as f:
    pickle.dump(dataset, f)

We can also look at the updated config fields, just to be sure.

In [12]:
config["imputers"]

[(['TotalCharges'], KNNImputer())]

In [11]:
config["numeric_scalers"]

[('tenure', None, RobustScaler()),
 ('MonthlyCharges', 'log1p', RobustScaler()),
 ('TotalCharges', 'log1p', RobustScaler())]